# WISER/BQP: Classical Physics-Informed Neural Network for Burgers’ Equation

This notebook develops and validates a classical Physics-Informed Neural Network (PINN) for the one-dimensional viscous Burgers equation

\[
\frac{\partial u}{\partial t}
+u\frac{\partial u}{\partial x}
=\nu\frac{\partial^2u}{\partial x^2},
\qquad x\in[-1,1],\quad t\in[0,1],
\]

with

\[
u(x,0)=-\sin(\pi x),
\qquad u(-1,t)=u(1,t)=0.
\]

The corrected PINN uses

\[
\hat u_\theta(x,t)
=-\sin(\pi x)+t(1-x^2)N_\theta(x,t),
\]

so the initial and boundary conditions are imposed exactly. The notebook generates the reference solution, reproducible collocation points, training histories, error metrics, figures, checkpoints, metadata, and a reproducibility archive.


## 1. Runtime and repository setup

For Google Colab, select **Runtime → Change runtime type → GPU**. Replace `YOUR_USERNAME` after the repository has been published.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/wiser-bqp-qapinn-cfd.git"
REPO_DIR = Path("/content/wiser-bqp-qapinn-cfd")

# In Colab, uncomment these lines after replacing YOUR_USERNAME:
# if not REPO_DIR.exists():
#     subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
# os.chdir(REPO_DIR)
# subprocess.run(["python", "-m", "pip", "install", "-q", "-e", "."], check=True)

print("Working directory:", Path.cwd())


## 2. Confirm the Python and GPU environment


In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Smoke test

The smoke profile checks the complete workflow. It is not the final accuracy result.


In [ ]:
RUN_SMOKE = True

if RUN_SMOKE:
    subprocess.run([
        "python", "scripts/run_burgers_cpinn.py",
        "--profile", "smoke",
        "--seed", "0",
        "--output-root", "outputs",
    ], check=True)


## 4. Smooth Burgers baseline

The validated smooth problem uses

\[

u=
rac{0.1}{\pi}.
\]


In [ ]:
RUN_BASELINE_SEED_0 = False

if RUN_BASELINE_SEED_0:
    subprocess.run([
        "python", "scripts/run_burgers_cpinn.py",
        "--profile", "baseline",
        "--seed", "0",
        "--output-root", "outputs",
    ], check=True)


## 5. Reproducibility seeds

Keep the architecture, collocation procedure, precision, and optimizer settings fixed.


In [ ]:
RUN_BASELINE_SEEDS_1_AND_2 = False

if RUN_BASELINE_SEEDS_1_AND_2:
    for seed in [1, 2]:
        subprocess.run([
            "python", "scripts/run_burgers_cpinn.py",
            "--profile", "baseline",
            "--seed", str(seed),
            "--output-root", "outputs",
        ], check=True)


## 6. Aggregate the results


In [ ]:
import pandas as pd
from IPython.display import display

summary_path = Path("outputs/experiment_summary.csv")
if summary_path.exists():
    frame = pd.read_csv(summary_path)
    selected = frame[
        (frame["profile"] == "baseline")
        & (frame["viscosity_label"] == "nu_0p1pi")
    ].copy()
    display(selected[[
        "seed",
        "relative_l2_error",
        "pde_residual_rms",
        "total_training_seconds",
        "trainable_parameters",
    ]])
    display(selected[[
        "relative_l2_error",
        "pde_residual_rms",
        "total_training_seconds",
    ]].agg(["mean", "std", "min", "max"]))
else:
    print("No experiment summary found yet.")


## 7. Low-viscosity Burgers case

Run this only after the smooth baseline is stable across multiple seeds:

\[

u=
rac{0.01}{\pi}.
\]


In [ ]:
RUN_LOW_VISCOSITY = False

if RUN_LOW_VISCOSITY:
    subprocess.run([
        "python", "scripts/run_burgers_cpinn.py",
        "--profile", "main",
        "--seed", "0",
        "--output-root", "outputs",
    ], check=True)


## 8. Next controlled experiments

After the three-seed classical baseline:

1. parameter-matched smaller classical PINN;
2. classical Fourier-feature PINN;
3. 3-, 4-, and 5-qubit QAPINNs;
4. Fourier, gradient, kernel, loss-landscape, and entanglement analyses;
5. heat-equation and reduced projectile-CFD benchmarks.
